<a href="https://colab.research.google.com/github/Lorena-garciamorais/2026_Intro_Python/blob/main/live/part-I/1.6-geospatial-vector-data-practice-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.6-geospatial-vector-data-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [52]:
%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Practice Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [53]:
# Your solution here
df = pd.DataFrame(
    {"Name" : ["A", "B"],
     "lon" : [7.4, 8.5],
     "lat" : [46.9, 47.4],

    }
)

gdf = gpd.GeoDataFrame(
    df,
    geometry = gpd.points_from_xy(df["lon"], df["lat"]),
    crs = "EPSG:4326",
)

gdf


,Name,lon,lat,geometry
0,A,7.4,46.9,POINT (7.4 46.9)
1,B,8.5,47.4,POINT (8.5 47.4)


In [54]:
print(gdf.geometry)
print("crs", gdf.crs)

0    POINT (7.4 46.9)
1    POINT (8.5 47.4)
Name: geometry, dtype: geometry
crs EPSG:4326


## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [55]:
from pathlib import Path
from shapely.geometry import Point, LineString, MultiPolygon

In [56]:
# Your solution here
Path("_files").mkdir(exist_ok=True)

gpd.GeoDataFrame(
    {"Name": ["A", "B"]},
    geometry=[Point(7.4, 46.9),
              Point(8.5, 47.4)],
    crs="EPSG:4326",
).to_file("_files/points.gpkg", driver="GPKG")

In [57]:
points = gpd.read_file("_files/points.gpkg")
points

,Name,geometry
0,A,POINT (7.4 46.9)
1,B,POINT (8.5 47.4)


In [58]:
print(points.shape)
print("crs", points.crs)
print("crs", points.crs.to_epsg())
# .crs --> donne EPSG : code
# .crs.to_epsg() --> donne que le code

(2, 2)
crs EPSG:4326
crs 4326


## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [59]:
# Your solution here
points_lv95 = points.to_crs(2056)
print("before:", points.crs.to_epsg(), "->", "after:", points_lv95.crs.to_epsg())

before: 4326 -> after: 2056


In [60]:
# projection en deg
print("A WGS84 (deg):", points.geometry.x.iloc[0], points.geometry.y.iloc[0])

# reprojection en m
print("A LV95 (m):  ", round(points_lv95.geometry.x.iloc[0], 1), round(points_lv95.geometry.y.iloc[0], 1))

A WGS84 (deg): 7.4 46.9
A LV95 (m):   2597056.2 1194321.9


## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [61]:
# Your solution here
dist_m = points_lv95.geometry.distance(points_lv95.geometry.iloc[1])
print("A -> B distance:", round(dist_m.iloc[0] / 1000, 1), "km")

A -> B distance: 100.2 km


## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [69]:
# Your solution here

from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")

point_inside = gpd.sjoin(points, poly, predicate="within", how="inner")
print("Point inside zone:", point_inside["Name"].tolist())

Point inside zone: ['A']


## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [83]:
# Your solution here
buffers = points_lv95.buffer(10000) # en mètre
print("buffer area :", round(buffers.area.iloc[0] / 1e6), "km^2")

buffer area : 314 km^2


## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here
